"Can I convert unstructured text into useful machine-learning features?"

In [0]:
"Coordinated pump-and-dump activity led to rapid price increase before large sell-off."

'Coordinated pump-and-dump activity led to rapid price increase before large sell-off.'

In [0]:

from pyspark.sql import functions as F

df_narratives = spark.table(
    "hackathon.shared_datasets.fraud_busters_narratives_enriched"
)

print(f"Rows: {df_narratives.count():,}")
print(f"Columns: {len(df_narratives.columns)}")

display(df_narratives.limit(5))


Rows: 1,000
Columns: 10


label,narrative,price_change_pct,social_mention_spike_x,timestamp,token,top10_holder_pct,tx_id,unique_buy_wallets,volume_spike_x
fraud,Anonymous channel activity signalled entry and KRUX spiked rapidly; 69% of supply moved toward exchange deposit addresses shortly after the rally.,177,19,2026-03-01T06:05:00.000Z,KRUX,59,TX00001,1506,65
fraud,"A low-liquidity OBLIX pool repriced after a large buy sequence, followed by rapid distribution across 55 newly active wallets.",284,18,2026-03-01T12:48:00.000Z,OBLIX,69,TX00002,488,50
fraud,Anonymous channel activity signalled entry and GLIMR spiked rapidly; 41% of supply moved toward exchange deposit addresses shortly after the rally.,522,16,2026-03-01T20:38:00.000Z,GLIMR,60,TX00003,1773,45
fraud,"A cluster of recently created wallets accumulated 31% of FENRA supply, then sold in a coordinated 25-minute window.",578,15,2026-03-02T05:04:00.000Z,FENRA,69,TX00004,2261,20
fraud,Liquidity tied to original deployer wallets was withdrawn shortly after a scripted social campaign accelerated retail buying in ZYNC.,503,19,2026-03-02T08:10:00.000Z,ZYNC,54,TX00005,1565,32


In [0]:
df_features = (
    df_narratives
    .withColumn(
        "narrative_length",
        F.length("narrative")
    )
    .withColumn(
        "word_count",
        F.size(
            F.split(
                F.col("narrative"),
                " "
            )
        )
    )
)

In [0]:
risk_keywords = [

    "wallet",
    "exchange",
    "liquidity",
    "deployer",
    "supply",
    "coordinated",
    "social",
    "campaign",
    "buying",
    "selling",
    "cluster",
    "active_wallets"


]


keyword_mapping = {
    "wallet": "wallet",
    "exchange": "exchange",
    "liquidity": "liquidity",
    "deployer": "deployer",
    "supply": "supply",
    "coordinated": "coordinated",
    "social": "social",
    "campaign": "campaign",
    "buying": "buying",
    "selling": "selling",
    "cluster": "cluster",
    "active_wallets": "active wallets"
}


In [0]:

for col_name, search_term in keyword_mapping.items():

    df_features = df_features.withColumn(
        f"kw_{col_name}",
        F.when(
            F.lower(F.col("narrative")).contains(search_term),
            1
        ).otherwise(0)
    )


In [0]:

semantic_components = [
    "pump_dump_flag",
    "rug_pull_flag",
    "coordination_flag",
    "wallet_activity_flag"
]

df_features = df_features.withColumn(
    "pump_dump_flag",
    F.when(
        F.lower(F.col("narrative")).rlike("pump|dump"),
        1
    ).otherwise(0)
)

df_features = df_features.withColumn(
    "rug_pull_flag",
    F.when(
        F.lower(F.col("narrative")).rlike("rug|liquidity"),
        1
    ).otherwise(0)
)

df_features = df_features.withColumn(
    "coordination_flag",
    F.when(
        F.lower(F.col("narrative")).rlike("coordination|coordinated|group"),
        1
    ).otherwise(0)
)

df_features = df_features.withColumn(
    "wallet_activity_flag",
    F.when(
        F.lower(F.col("narrative")).contains("wallet"),
        1
    ).otherwise(0)
)

df_features = df_features.withColumn(
    "insider_flag",
    F.when(
        F.lower(F.col("narrative")).contains("insider"),
        1
    ).otherwise(0)
)


In [0]:

display(
    df_features.select(
        "pump_dump_flag",
        "rug_pull_flag",
        "coordination_flag",
        "wallet_activity_flag",
        "insider_flag"
    ).limit(5)
)


pump_dump_flag,rug_pull_flag,coordination_flag,wallet_activity_flag,insider_flag
0,0,0,0,0
0,1,0,1,0
0,0,0,0,0
0,0,1,1,0
0,1,0,1,0


In [0]:
print(df_features.columns)

['label', 'narrative', 'price_change_pct', 'social_mention_spike_x', 'timestamp', 'token', 'top10_holder_pct', 'tx_id', 'unique_buy_wallets', 'volume_spike_x', 'narrative_length', 'word_count', 'kw_wallet', 'kw_exchange', 'kw_liquidity', 'kw_deployer', 'kw_supply', 'kw_coordinated', 'kw_social', 'kw_campaign', 'kw_buying', 'kw_selling', 'kw_cluster', 'kw_active_wallets', 'pump_dump_flag', 'rug_pull_flag', 'coordination_flag', 'wallet_activity_flag', 'insider_flag']


In [0]:

semantic_features = [

    "price_change_pct",
    "volume_spike_x",

    "social_mention_spike_x",

    "top10_holder_pct",

    "unique_buy_wallets",

    "narrative_length",

    "word_count",

    "risk_keyword_count",

    "pump_dump_flag",

    "rug_pull_flag",

    "coordination_flag",

    "wallet_activity_flag",

    "semantic_risk_score"

]


In [0]:

(
    df_features
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "hackathon.shared_datasets.fraud_busters_semantic_features"
    )
)

print("✅ Semantic features saved")


✅ Semantic features saved


In [0]:
# ============================================================
# Semantic Feature Comparison Table
# Fraud vs Legitimate Narratives
# ============================================================

from pyspark.sql import functions as F
import pandas as pd

# ------------------------------------------------------------
# 1. Define semantic keyword columns
# ------------------------------------------------------------

semantic_keyword_cols = [
    c for c in df_features.columns
    if c.startswith("kw_")
]

# Add overall keyword count if it exists
if "risk_keyword_count" in df_features.columns:
    semantic_keyword_cols = ["risk_keyword_count"] + semantic_keyword_cols

print("Semantic columns used:")
for c in semantic_keyword_cols:
    print(c)

# ------------------------------------------------------------
# 2. Create average feature values by label
# ------------------------------------------------------------

summary_df = (
    df_features
    .groupBy("label")
    .agg(
        *[
            F.round(F.avg(F.col(c)), 3).alias(c)
            for c in semantic_keyword_cols
        ] )
)

display(summary_df)

Semantic columns used:
kw_wallet
kw_exchange
kw_liquidity
kw_deployer
kw_supply
kw_coordinated
kw_social
kw_campaign
kw_buying
kw_selling
kw_cluster
kw_active_wallets


label,kw_wallet,kw_exchange,kw_liquidity,kw_deployer,kw_supply,kw_coordinated,kw_social,kw_campaign,kw_buying,kw_selling,kw_cluster,kw_active_wallets
fraud,0.88,0.12,0.282,0.143,0.28,0.297,0.143,0.143,0.143,0.0,0.302,0.138
legitimate,0.398,0.118,0.0,0.0,0.13,0.0,0.0,0.125,0.0,0.0,0.143,0.0
